## 1. RDD(Resilient Distributed DataSet)
## 
### Definition -> Its collection of object like list in python , its immutable and in memory processing


###  RDD came first—DataFrames were created later to improve and replace direct RDD usage for most tasks.\
### 
### Neither replaced the standard Python list. Here is the chronological evolution of how these tools developed:\

**Timeline of Evolution
**Python List: A basic data structure built into Python that stores data in the RAM of a single computer.\

RDD (Resilient Distributed Dataset): Created when Apache Spark was first developed. Spark needed a way to distribute Python/Scala objects across a cluster of multiple computers in memory. RDD was the original data structure of Spark.\

DataFrame: Introduced later in Spark to improve performance over RDDs. DataFrames added named columns, schemas, and automatic execution optimization.

In [0]:
data = [('Bhupesh' , 23) , ('Shivani' , 24)]

rdd = spark.sparkContext.parallelize(data)
print(rdd.collect())

In [0]:
# we can transform rdd to data frame there are two ways

# 1. using rdd.toDF()
df = rdd.toDF(schema = ['name' , 'age'])
df.show()

# 2. using spark.createDataFrame()
df = spark.createDataFrame(rdd , schema = ['name' , 'age'])
df.show()

## 2. map()
## 
### When you use map(), you are taking your data—which is split into multiple pieces (partitions) and spread across multiple computers (worker nodes) in a cluster—and telling Spark:
### 
### Take this function and run it on every single piece of data across all computers in the cluster at the same time.

## 3. flatMap()

## 4. partition() 
## 
### A partition is a logical chunk of a large dataset. When you create an RDD or DataFrame, Spark splits the data into multiple partitions and distributes them across the worker nodes in your cluster so they can be processed simultaneously in parallel.


**Divide the data onto basis of one or multipe column**

In [0]:
data  = [(1 , 'Bhupesh' , 'IT') , (2 , 'Shivani' , 'HR')]
schema = ['id' , 'name' , 'department'] 

df = spark.createDataFrame(data = data , schema = schema)

df.write.parquet('/Volumes/workspace/xyz/employee' , mode ='overwrite' , partitionBy='department')


In [0]:
df1 = spark.read.parquet('/Volumes/workspace/xyz/employee/department=HR')
df1.show()

## 5. Date Function
## 
**current_date() - get the current system date by default the data will be returned in yyyy-dd-mm\

to_date() - convert date string into dateType\

 date_format()  - to convert the yyyy-dd-mm  from  to specified format**

In [0]:
df = spark.range(3)
df.show()


from pyspark.sql.functions import current_date  , date_format , col

df1 = df.withColumn('current_date' , current_date())
df1.show()



df2= df1.withColumn('current_date' , date_format(col('current_date') , 'dd-MM-yyyy'))
df2.show()

datediff()  -  diffrene of date 

months_between  - function calculates the difference in months between two dates or timestamps in PySpark

add_months() , add_year()-  adding months , year

for substarct use -x = i  front of value

In [0]:
df = spark.createDataFrame([('2016-06-22'  , '2016-06-01')] , ['d1', 'd2'])

df.show()


from pyspark.sql.functions import datediff , months_between , add_months , col

df1 = df.withColumn('datediff' , datediff(col('d1') , col('d2'))).show()

df2 = df.withColumn('months_between' , months_between(col('d1') , col('d2'))).show()

df3 = df.withColumn('month_add' , add_months(col('d1') , 2)).show()


df4 = df.withColumn('month_sub' , add_months(col('d1') , -1)).show()




currentTimeStamp()

In [0]:
from pyspark.sql.functions import  currrent_timestamp

df =spark.range(3)

df1= df.withColumn('timestamp', current_timestamp())
df1.show(truncate=False)

## 6. Aggregation Function

1) approx_count_distinct  -  return count of disticnt values

2) avg()  - return avg of values in a group of rows

3) collect_list() -  return all values of column in a list duplicates as well

4) collect_set() = return all values of colmn in a list but distinct

In [0]:
data  =[(1 , 'Bhupesh' , 'IT' , 15000) , (2 , 'Shivani' , 'HR' , 20000) , (3 , 'Ravi' , 'IT' , 15000)]
schema = ['id' , 'name' , 'department', 'salary'] 

df = spark.createDataFrame(data = data , schema = schema)
df.show()



from pyspark.sql.functions import *


df1 = df.select(approx_count_distinct(col('salary')))
df1.show()

df2 = df.select(avg('salary')).show()

df3 = df.select(collect_list(col('salary')))
df3.show()

df4 = df.select(collect_set(col('salary')))
df4.show()

## 7. Ranking Function

### Window.partitionBy() -  is used to divide your dataset into isolated groups (buckets) so that window calculations happen independently inside each group rather than across the whole DataFrame.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, dense_rank, max as spark_max, lag
)


# 1. Sample Employee Data
data = [
    (101, "Bhupesh", "Data Science", 120000),
    (102, "Shivani", "Data Science", 135000),
    (103, "Amit", "Data Science", 95000),
    (104, "Pooja", "HR", 70000),
    (105, "Ravi", "HR", 85000),
    (106, "Neha", "HR", 85000),
    (107, "Karan", "Engineering", 110000)
]

schema = ["emp_id", "name", "department", "salary"]
df = spark.createDataFrame(data, schema)

# 2. Define Window Specifications
# Window A: Partition by department, ordered by salary descending (for ranking & lag)
window_dept_salary = Window.partitionBy("department").orderBy(col("salary").desc())

# Window B: Partition by department only (for department-wide max salary)
window_dept_only = Window.partitionBy("department")

# 3. Apply Window Transformations
df_transformed = df \
    .withColumn("dept_rank", dense_rank().over(window_dept_salary)) \
    .withColumn("max_dept_salary", spark_max("salary").over(window_dept_only)) \
    .withColumn("salary_gap_from_top", col("max_dept_salary") - col("salary")) \
    .withColumn("next_higher_salary", lag("salary", 1).over(window_dept_salary))

# 4. Filter for Top 2 Earners Per Department
df_top2 = df_transformed.filter(col("dept_rank") <= 2)

df_top2.select(
    "department", "dept_rank", "name", "salary", 
    "max_dept_salary", "salary_gap_from_top", "next_higher_salary"
).show()